# tSCS EMG — 30 Hz burst trains — polarity × lidocaine (subject NTA, 24-07-2026)

One participant, one session, a **2 × 2 design**: stimulation polarity (**cathodic** = polarity 2,
the usual cathode-on-the-spine montage; **anodic** = polarity 1) × lidocaine (**before** / **with**).
Electrode 2, anode at the iliac crests, folder `testSCS` (`changedpol.xlsx`).


## What is measured — fully automatic
For each train the first `N_PULSES` pulses are analysed: `window_k = [pulse_k onset +
RESP_START_MS, pulse_(k+1) onset − GUARD_MS]`, `p2p_k = max − min` in it (mV). A train counts as
a **motor response** only if pulse-1 p2p ≥ `MIN_SNR` × the pre-stimulus baseline p2p; a train whose
max/min sit on the window borders in more than `MAX_EDGE_FRAC` of its pulses is **rejected as
artifact** (smooth artifact recovery, no EMG wave). Pulses whose window contains a sensor dropout
(exact zeros) are skipped. **Nothing is averaged across intensities.**

## Colours and styles, everywhere
**gray = before lidocaine, orange = with lidocaine** — the same two colours as in the original
notebooks. **Polarity is the style: cathodic = solid lines / plain bars / filled markers,
anodic = dashed lines / hatched bars / hollow markers.** So colour answers "lidocaine?", style
answers "which polarity?".

## Files
| | single pulse | burst | ARC-EX (Modulated) |
|---|---|---|---|
| **cathodic · before** | `100913` (10:09) | `102443` (10:24; `102236` aborted) | `103530` (10:35; `103040` no response) |
| **cathodic · lidocaine** | `113447` (11:34) | `113834` (11:38) | `114332` (11:43) |
| **anodic · before** | `101941` (10:19) | `102721` (10:27) | `103849` (10:38) |
| **anodic · lidocaine** | `113632` (11:36) | `114055` (11:40) | `114730` (11:47; `114630` ignore) |

Motor threshold from the log — burst: cathodic 25 mA before / 30 lidocaine, anodic 30 / 30;
ARC-EX: cathodic 70 / 70, anodic 65 / 90; single pulse ~30–40. Participant: *"less pain with
anodic"*, *"much less pain after lidocaine"*. Lidocaine applied 45 min (~10:50 → 11:33).

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, detect_pulses, pretty, waterfall, waterfall_overlay
from functions.burst import (diagnostics, compare_at_intensity, summary_curves, burst_p2p,
                             save_burst_csv, plot_pulse_overlay, detection_report)
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
COND = {   # label: (file, motor threshold mA from the log)
    "cathodic · before":    ("Burst_autosave_20260724_102443_670ms.csv", 25),
    "cathodic · lidocaine": ("Burst_autosave_20260724_113834_522ms.csv", 30),
    "anodic · before":      ("Burst_autosave_20260724_102721_694ms.csv", 30),
    "anodic · lidocaine":   ("Burst_autosave_20260724_114055_891ms.csv", 30),
}
COLOURS = {"cathodic · before": "0.45", "cathodic · lidocaine": "#f39c12",     # colour = lidocaine
           "anodic · before": "0.45", "anodic · lidocaine": "#f39c12"}
STYLE = {   # style = polarity: (bar hatch, line style, curve marker)
    "cathodic": ("",    "-",  "o"),
    "anodic":   ("///", "--", "s"),
}
def style(keys, what):    # what: "hatch" | "ls" | "marker" | "hollow"
    i = {"hatch": 0, "ls": 1, "marker": 2}.get(what)
    return [STYLE[k.split(" · ")[0]][i] if what != "hollow" else k.startswith("anodic") for k in keys]

N_PULSES      = 10     # first N pulses of each train
RESP_START_MS = 8.0    # response window starts this long after EACH pulse onset (must clear the artifact)
                       # can be a dict to override single channels, e.g. {"R_DELmed": 11.0}
GUARD_MS      = 1.0    # ...and stops this long before the next pulse
SNR_ON        = "median" # which pulses the criterion looks at: "median" (the train's median
                        # pulse), "p1" (pulse 1 only), "half" (at least half the pulses).
                        # "p1" throws away a whole train when its first pulse happens to be small.
MIN_SNR       = 1.2    # motor-response criterion (None = keep all)
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0   # anchor each pulse's max/min to the train average
MAX_EDGE_FRAC = 0.5    # artifact rejection (None = off)
EDGE_MS       = 1.0    # "on a window border" means within this many ms of it
JITTER_MS     = 0.5    # flag a pulse whose peak latency differs from the train
                       # median by more than this (floored at the anchor window)
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)

LABELS = list(COND)
CSVS   = [D + COND[k][0] for k in LABELS]
MTS    = [COND[k][1] for k in LABELS]
COLS   = [COLOURS[k] for k in LABELS]
RUNS   = [load_run(f) for f in CSVS]                                      # [(meta, t, sig), ...]
muscles = [c for c in RUNS[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS)]
AMPS    = sorted(set.intersection(*[{m["amp_ma"] for m in r[0]} for r in RUNS]))   # in ALL four files
for lab, f, (meta, t, sig) in zip(LABELS, CSVS, RUNS):
    print(f"{lab:22s} {f.split('/')[-1]:48s} {[m['amp_ma'] for m in meta]} mA")
print("common intensities:", AMPS, "mA  | motor thresholds:", dict(zip(LABELS, MTS)))

def sel(*keys):        # helper: files / labels / colours / MTs of a subset of conditions, in order
    return ([D + COND[k][0] for k in keys], list(keys), [COLOURS[k] for k in keys], tuple(COND[k][1] for k in keys))
HATCH, LS, MARK = style(LABELS, "hatch"), style(LABELS, "ls"), style(LABELS, "marker")


## 2 · Diagnostics — is the detection sound? (one block per condition)

Artifact width per channel (`RESP_START_MS` must clear it), trains passing the motor-response
criterion and from which mA, artifact-rejected trains, sensor dropouts, clipped channels.

In [ ]:
RES = []
for lab, f, (meta, t, sig), mt in zip(LABELS, CSVS, RUNS, MTS):
    print("=" * 26, lab, "=" * 26)
    amp_show = mt if mt in [m["amp_ma"] for m in meta] else max(m["amp_ma"] for m in meta)
    RES.append(diagnostics(meta, t, sig, muscles, amp=amp_show, **KW))
    print("saved", save_burst_csv(RES[-1], muscles, f, meta=meta, normalize="none")); print()


## 3 · Reliability — pulse overlays at each condition's motor threshold

Every pulse re-aligned to its own onset and overlaid; stacked ▼/▲ = the same deflection is
detected every time; red rings = flagged (on a window border, or at a different latency than the
other pulses). Then the numeric report per condition.

In [ ]:
for lab, (meta, t, sig), mt in zip(LABELS, RUNS, MTS):
    amp_show = mt if mt in [m["amp_ma"] for m in meta] else max(m["amp_ma"] for m in meta)
    plot_pulse_overlay(meta, t, sig, muscles, amp=amp_show, title=f"{lab} - {amp_show} mA", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW)


In [ ]:
for lab, res in zip(LABELS, RES):
    print("=" * 26, lab, "=" * 26); detection_report(res, muscles, edge_ms=EDGE_MS, jitter_ms=JITTER_MS); print()


## 4 · Raw traces — waterfalls

Per polarity: before, with lidocaine (same gain per muscle), and the two overlaid.

In [ ]:
XLIM_WF = (-20, 130)
for pol in ("cathodic", "anodic"):
    files, labs, cols, _ = sel(f"{pol} · before", f"{pol} · lidocaine")
    runs = [load_run(f) for f in files]
    print(f"===== {labs[0]}"); g = waterfall(*runs[0], muscles, xlim=XLIM_WF)
    print(f"===== {labs[1]}"); waterfall(*runs[1], muscles, xlim=XLIM_WF, gains=g)
    print(f"===== overlay"); waterfall_overlay(runs, muscles=muscles, xlim=XLIM_WF, gains=g, labels=labs, colours=cols)


## 5 · Every common intensity — all four conditions

Per muscle: the four traces overlaid with the detected ▼/▲, per-pulse peak-to-peak in **mV**
(four bars per pulse), and pulse 1 vs mean of pulses 2–N. *below criterion* / *ARTIFACT* say why a
train is left out. Figures in increasing mA; `NORM_BARS = "before_first"` gives bars as % of the
first condition's pulse 1 instead of mV.

In [ ]:
NORM_BARS = "none"
for a in AMPS:
    compare_at_intensity(CSVS, None, amp=a, normalize=NORM_BARS, labels=LABELS, colours=COLS,
                         hatches=HATCH, linestyles=LS, title=f"{a} mA", **KW)


## 6 · Across intensities — recruitment curves

Top: pulse-1 p2p (● solid) and mean of pulses 2–N (▲ dashed) in mV; hollow = rejected.
Bottom: mean 2–N as % of pulse 1 (depression along the train).
**6a** lidocaine within each polarity · **6b** polarity before and with lidocaine · **6c** all four.

In [ ]:
for pol in ("cathodic", "anodic"):                                   # 6a - lidocaine, per polarity
    files, labs, cols, _ = sel(f"{pol} · before", f"{pol} · lidocaine")
    summary_curves(files, None, labels=labs, colours=cols, markers=style(labs, "marker"), **KW)


In [ ]:
for state in ("before", "lidocaine"):                                # 6b - polarity, per state
    files, labs, cols, _ = sel(f"cathodic · {state}", f"anodic · {state}")
    summary_curves(files, None, labels=labs, colours=cols, markers=style(labs, "marker"), **KW)


In [ ]:
summary_curves(CSVS, None, labels=LABELS, colours=COLS, markers=MARK, **KW);        # 6c - all four


## 7 · At motor threshold — all four, each at its own MT

Same figure as §5 but with each condition at the motor threshold from the log (they differ), so the
four are compared at matched *relative* intensity.

In [ ]:
compare_at_intensity(CSVS, None, amp=tuple(MTS), normalize="none", labels=LABELS, colours=COLS,
                     hatches=HATCH, linestyles=LS, title="each condition at its own motor threshold", **KW)


## 8 · Focus — one muscle, your choice of intensity per condition

Pick the muscle and, for each condition you want to show, the intensity. Remove a line from
`PICK` to leave that condition out. The pulse overlay of each chosen train follows, to check the
peaks.

In [ ]:
MUSCLE = "Flex. digitorum (R)"          # label from any panel title, or a channel name
PICK = {                                 # condition: intensity (mA)
    "cathodic · before":    25,
    "cathodic · lidocaine": 30,
    "anodic · before":      30,
    "anodic · lidocaine":   30,
}
NORM = "none"                            # "none" = mV | "before_first" = % of the first listed condition's pulse 1

files, labs, cols, _ = sel(*PICK)
compare_at_intensity(files, None, amp=tuple(PICK.values()), normalize=NORM, muscles=MUSCLE,
                     labels=labs, colours=cols, hatches=style(labs, "hatch"), linestyles=style(labs, "ls"),
                     title=MUSCLE, **KW)
for k in PICK:
    meta_, t_, sig_ = load_run(D + COND[k][0])
    plot_pulse_overlay(meta_, t_, sig_, MUSCLE, amp=PICK[k], title=f"{k} - {PICK[k]} mA", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW)
